In [1]:
import pandas as pd
from genai.client import Client
from genai.credentials import Credentials
from genai.schema import (
    TextGenerationParameters,
    TextGenerationReturnOptions,
    DecodingMethod
)
from dotenv import load_dotenv
from transformers import AutoModelForCausalLM, AutoTokenizer
import ray
import json
import numpy as np
import random

/Users/christodoulosconstantinides/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-04-06 18:06:49,895	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
load_dotenv()

True

In [3]:
ray.init(num_cpus=8)

2024-04-06 18:06:53,579	INFO worker.py:1715 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


Python version:,3.10.13
Ray version:,2.9.0
Dashboard:,http://127.0.0.1:8265


In [5]:
# train_df = pd.read_csv('../fmea_result/autoQ_train_data_input_for_experiments.csv')
# val_df = pd.read_csv('../fmea_result/autoQ_val_data_input_for_experiments.csv')
# test_df = pd.read_csv('../fmea_result/autoQ_test_data_input_for_experiments.csv')

In [4]:
train_df = pd.read_csv('processed/mixtral_train_with_entities.csv')
val_df = pd.read_csv('processed/mixtral_val_with_entities.csv')
test_df = pd.read_csv('processed/mixtral_test_with_entities.csv')

In [5]:
train_df['mode'] = 'train'
val_df['mode'] = 'val'
test_df['mode'] = 'test'

In [6]:
df = pd.concat([train_df, val_df, test_df])

In [7]:
num_shuffle = 3
num_missing = 3

In [76]:
def shuffle_failure_locations(df):
    result = []
    for i in range(len(df)):
        row = df.iloc[i]
        failure_locations = list(eval(row['failure_locations']))
        for j in range(num_shuffle):
            shuffled_locations = failure_locations.copy()
            random.shuffle(shuffled_locations)
            df.loc[i, f'failurelocation_shuffle_{j}'] = str(shuffled_locations)
            result.append(row)
    return pd.DataFrame(result)

def drop_failure_locations(df):
    for i in range(len(df)):
        row = df.iloc[i]
        failure_locations = list(eval(row['failure_locations']))
        for j in range(num_missing):
            missing_items = random.choice(failure_locations)
            remaining_items = failure_locations.copy()
            remaining_items.remove(missing_items)
            df.loc[i, f'failure_location_drop_info_{j}'] = str(missing_items)
            df.loc[i, f'failure_location_drop_{j}'] = str(remaining_items)

def drop_boundary_locations(df):
    for i in range(len(df)):
        for j in range(num_missing):
            row = df.iloc[i]
            long_description = row.long_description
            parsed_entities = eval(row.parsed_entities)
            if len(parsed_entities) == 0:
                continue
            to_drop = random.choice(parsed_entities)
            df.loc[i, f'assetlongdescription_entity_drop_info_{j}'] = long_description.replace(to_drop, '')
            df.loc[i, f'assetlongdescription_entity_drop_{j}'] = to_drop

def random_split_list(df, col_name):
    for i in range(len(df)):
        for j in range(num_shuffle):
            items = list(eval(df[col_name].iloc[i]))
            if len(items) <= 1:
                continue
            n_items = len(items)
            random.shuffle(items)
            left_n_items = max(1, random.randint(1, n_items-1))
            df.loc[i, f'{col_name}_left_{j}'] = str(items[:left_n_items])
            df.loc[i, f'{col_name}_right_{j}'] = str(items[left_n_items:])

def augment_df(df):
    shuffle_failure_locations(df)
    drop_failure_locations(df)
    drop_boundary_locations(df)
    # return pd.concat([shuffled_failure_locations, missing_failure_locations, missing_boundary_locations])

### Asset2Item

In [77]:
df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
augment_df(df)

In [78]:
col_rename = {
    'TypeData.CompTypeID': 'asset_id',
    'TypeData.GenCompType': 'asset_name',
    'long_description': 'assetlongdescription_original',
    'component_short_description': 'assetshort_description_original',
    'failure_locations': 'failurelocation_original',
    'parsed_entities': 'assetlongdescription_entity_llms'
}

In [79]:
df.drop(['answer', 'component_boundry_for_short_description', 'Unnamed: 0.1', 'Unnamed: 0'], axis=1, inplace=True)
df.rename(col_rename, axis=1, inplace=True)
# todo: assetfailuremodeinfo_llm, assetlongdescription_llm, assetshortdescription_llm

In [80]:
df.head(2)

,asset_id,asset_name,assetlongdescription_original,failurelocation_original,assetshort_description_original,assetlongdescription_entity_llms,mode,failurelocation_shuffle_0,failurelocation_shuffle_1,failurelocation_shuffle_2,...,failure_location_drop_info_1,failure_location_drop_1,failure_location_drop_info_2,failure_location_drop_2,assetlongdescription_entity_drop_info_0,assetlongdescription_entity_drop_0,assetlongdescription_entity_drop_info_1,assetlongdescription_entity_drop_1,assetlongdescription_entity_drop_info_2,assetlongdescription_entity_drop_2
0,Accumulator_2_V0,Accumulator - Pneumatic - Bladder Type,The equipment Accumulator - Pneumatic - Bladde...,"{'Gas Fill Valve', 'Bladder', 'Air line Check ...",Accumulator - Pneumatic - Bladder Type,"['Accumulator - Pneumatic - Bladder Type', 'Fi...",train,"['Bladder', 'Gas Fill Valve', 'Air line Check ...","['Air line Check Valve, if present', 'Gas Fill...","['Gas Fill Valve', 'Air line Check Valve, if p...",...,Bladder,"['Air line Check Valve, if present', 'Gas Fill...","Air line Check Valve, if present","['Bladder', 'Gas Fill Valve']",The equipment Accumulator - Pneumatic - Bladde...,Fixed Asset,The equipment Accumulator - Pneumatic - Bladde...,Fixed Asset,The equipment Accumulator - Pneumatic - Bladde...,Air Line Check Valve
1,AirEjector_SJ_1,Steam Jet Air Ejector,"The equipment Steam Jet Air Ejector, is catego...",{'Condenser - Internal Hardware; including: Ba...,Steam Jet Air Ejector,"['Steam Jet Air Ejector', 'Fixed Asset', 'Ejec...",train,"['Condenser - Tube Joint: Rolled', 'Ejector - ...","['Condenser - Shell, Inlet and Outlet Nozzles'...",['Condenser - Closure Devices Channel Partitio...,...,Condenser - Tube Sheets,"['Condenser - Tubes', 'Ejector - Steam Nozzle'...",Condenser - Closure Devices Channel Partitions...,"['Condenser - Tubes', 'Ejector - Steam Nozzle'...","The equipment Steam Jet Air Ejector, is catego...",Steam chest,"The equipment Steam Jet Air Ejector, is catego...",Steam nozzle,"The equipment Steam Jet Air Ejector, is catego...",Discharge port


In [81]:
df.to_csv('processed/asset2item.csv')

### Item2Item

In [82]:
col_name = 'failurelocation_original'
item2item_df = df[[col_name]].copy()
random_split_list(item2item_df, col_name)

In [83]:
item2item_df.head(2)

,failurelocation_original,failurelocation_original_left_0,failurelocation_original_right_0,failurelocation_original_left_1,failurelocation_original_right_1,failurelocation_original_left_2,failurelocation_original_right_2
0,"{'Gas Fill Valve', 'Bladder', 'Air line Check ...","['Air line Check Valve, if present']","['Bladder', 'Gas Fill Valve']",['Gas Fill Valve'],"['Air line Check Valve, if present', 'Bladder']","['Gas Fill Valve', 'Air line Check Valve, if p...",['Bladder']
1,{'Condenser - Internal Hardware; including: Ba...,['Condenser - Closure Devices Channel Partitio...,"['Condenser - Shell, Inlet and Outlet Nozzles'...",['Condenser - Internal Hardware; including: Ba...,"['Condenser - Tube Sheets', 'Condenser - Closu...","['Condenser - Tubes', 'Condenser - Tube Joint:...",['Condenser - Internal Hardware; including: Ba...


In [84]:
item2item_df.to_csv('processed/item2item.csv')

### Attribute2Attribute

In [85]:
col_name = 'assetlongdescription_entity_llms'
attribute2attribute_df = df[[col_name]].copy()
random_split_list(attribute2attribute_df, col_name)

In [86]:
attribute2attribute_df.head(2)

,assetlongdescription_entity_llms,assetlongdescription_entity_llms_left_0,assetlongdescription_entity_llms_right_0,assetlongdescription_entity_llms_left_1,assetlongdescription_entity_llms_right_1,assetlongdescription_entity_llms_left_2,assetlongdescription_entity_llms_right_2
0,"['Accumulator - Pneumatic - Bladder Type', 'Fi...","['Accumulator - Pneumatic - Bladder Type', 'Bl...","['Tank', 'Air Line Check Valve']","['Air Line Check Valve', 'Gas Precharge Valve'...","['Bladder', 'Fixed Asset']",['Accumulator - Pneumatic - Bladder Type'],"['Tank', 'Gas Precharge Valve', 'Air Line Chec..."
1,"['Steam Jet Air Ejector', 'Fixed Asset', 'Ejec...","['Ejector and internals', 'Valves', 'Discharge...","['Steam condensers, if present', 'Steam chest'...","['Ejector and internals', 'Steam inlet port', ...","['Steam chest', 'Steam condensers', 'Steam inl...","['Steam inlet port and nozzle, including steam...","['Steam Jet Air Ejector', 'Steam condensers', ..."


In [87]:
attribute2attribute_df.to_csv('processed/attribute2attribute.csv')